In [ ]:
import pandas_ta as ta
import pandas as pd

In [ ]:
# org_data = yf.download(tickers='^RUI', start='2014-03-11', end='2024-07-10')
# org_data.to_parquet('RUI.2014-03-11to2024-07-10.parquet')

In [ ]:
org_data = pd.read_parquet('RUI.2014-03-11to2024-07-10.parquet')
org_data

In [ ]:
org_data.index

In [ ]:
data = pd.DataFrame(index=org_data.index)
data.index.name = 'Date'
data['Adj Close'] = org_data['Adj Close']['^RUI']
data['Open'] = org_data['Open']['^RUI']
data['Close'] = org_data['Close']['^RUI']
data['High'] = org_data['High']['^RUI']
data['Low'] = org_data['Low']['^RUI']
'Freq=', (data.index.max() - data.index.min()) / len(data)

In [ ]:
from helper.importer import go

fig = go.Figure()
t = data
fig.add_trace(
    go.Candlestick(x=t.index, open=t['Open'], close=t['Close'], high=t['High'], low=t['Low']))
fig.show()

In [ ]:
full_date_range = pd.date_range(start=data.index.min(), end=data.index.max(), freq='D')
full_range_df = pd.DataFrame(index=full_date_range)
no_gap_data = full_range_df.merge(data, how='left', left_index=True, right_index=True)
no_gap_data.index.name = 'Date'
no_gap_data['index'] = range(1, len(no_gap_data) + 1)
no_gap_data

for column in set(no_gap_data.columns) & set(data):
    # Use `merge_asof` to get the next row where the current column is non-NA
    no_gap_data[[column]]
    data[[column]]
    no_gap_data[column] = pd.merge_asof(
        no_gap_data[[column]],  # the data to merge
        data[[column]],  # merging with rows that have no missing data
        left_index=True, right_index=True,
        direction='forward'  # Get the next non-NA value for each row
    )[column + '_y']
no_gap_data

In [ ]:
'Freq=', (no_gap_data.index.max() - no_gap_data.index.min()) / len(no_gap_data)

In [ ]:
from helper.importer import go

fig = go.Figure()
t = no_gap_data
fig.add_trace(
    go.Candlestick(x=t.index, open=t['Open'], close=t['Close'], high=t['High'], low=t['Low']))
fig.show()

In [ ]:
from profit_loss.profit_loss_adder import add_long_n_short_profit, plot_long_profit, plot_long_n_short_profit, \
    plot_short_profit

n2_data = add_long_n_short_profit(no_gap_data)


In [ ]:
plot_long_profit(n2_data)
plot_short_profit(n2_data)
plot_long_n_short_profit(n2_data)

In [ ]:
# position_max_days = 3 * 4 * 4 * 4 * 4  # 768 5mins = 64h = 2.66D
# action_delay = 2
# rolling_window = position_max_days - action_delay
# n2_data = no_gap_data

In [ ]:
# n2_data['worst_long_open'] = n2_data['High'].rolling(window=action_delay, min_periods=1).max().shift(1 - action_delay)
# n2_data['worst_short_open'] = n2_data['Low'].rolling(window=action_delay, min_periods=1).min().shift(1 - action_delay)
# n2_data['max_high'] = n2_data['High'].rolling(rolling_window, min_periods=rolling_window).max().shift(
#     -position_max_days)
# n2_data['min_low'] = n2_data['Low'].rolling(rolling_window, min_periods=rolling_window).min().shift(-position_max_days)
# 
# n2_data['max_high_distance'] = n2_data['High'].rolling(rolling_window).apply(lambda x: x.argmax(), raw=True).shift(
#     -position_max_days) + action_delay
# n2_data['min_low_distance'] = n2_data['Low'].rolling(rolling_window).apply(lambda x: x.argmin(), raw=True).shift(
#     -position_max_days) + action_delay

In [ ]:
# quantiles = 10
# for i in range(1, quantiles + 1):
#     q_rolling_window = int(i * rolling_window / quantiles)
#     n2_data[f'q{i}_max_high'] = \
#         n2_data['High'].rolling(q_rolling_window, min_periods=q_rolling_window).max().shift(-q_rolling_window)
#     n2_data[f'q{i}_min_low'] = \
#         n2_data['Low'].rolling(q_rolling_window, min_periods=q_rolling_window).min().shift(-q_rolling_window)
# 
#     n2_data[f'q{i}_max_high_distance'] = (n2_data['High'].rolling(q_rolling_window)
#                                           .apply(lambda x: x.argmax(), raw=True).shift(-q_rolling_window))
#     n2_data[f'q{i}_min_low_distance'] = (n2_data['Low'].rolling(q_rolling_window)
#                                          .apply(lambda x: x.argmin(), raw=True).shift(-q_rolling_window))

In [ ]:
# not_na_indexes = n2_data[~n2_data.isna().any(axis='columns')].index
# n2_data.loc[not_na_indexes, 'max_high_quantile'] = (
#         n2_data.loc[not_na_indexes, 'max_high_distance'] / (position_max_days / quantiles))
# n2_data.loc[not_na_indexes, 'min_low_quantile'] = (
#         n2_data.loc[not_na_indexes, 'min_low_distance'] / (position_max_days / quantiles))

In [ ]:
# import numpy as np
# 
# min_low_np_a = n2_data.loc[not_na_indexes, [f'q{i}_min_low' for i in range(1, quantiles + 1)]].to_numpy()
# quantile_max_high_consistency = n2_data.loc[not_na_indexes, 'max_high_quantile'].astype(int).eq(0)
# consistent_quantile_max_high = quantile_max_high_consistency[~quantile_max_high_consistency].index
# n2_data.loc[not_na_indexes, 'quantile_long_min_low'] = min_low_np_a[
#     np.arange(len(n2_data.loc[not_na_indexes])), n2_data.loc[not_na_indexes, 'max_high_quantile'].astype(int)]
# # n2_data.loc[consistent_quantile_max_high, 'pre_quantile_long_min_low'] = min_low_np_a[
# #     np.arange(len(consistent_quantile_max_high)), n2_data.loc[consistent_quantile_max_high, 'max_high_quantile'].astype(
# #         int) - 1]
# # n2_data.loc[consistent_quantile_max_high, 'long_drawdown_consistency'] = (
# #         n2_data.loc[consistent_quantile_max_high, 'pre_quantile_long_min_low']
# #         == n2_data.loc[consistent_quantile_max_high, 'quantile_long_min_low'])
# n2_data['long_drawdown'] = \
#     (n2_data['worst_long_open'] - n2_data['quantile_long_min_low']) / n2_data['worst_long_open']
# max_high_np_a = n2_data.loc[not_na_indexes, [f'q{i}_max_high' for i in range(1, quantiles + 1)]].to_numpy()
# quantile_min_low_consistency = n2_data.loc[not_na_indexes, 'min_low_quantile'].astype(int).eq(0)
# consistent_quantile_min_low = quantile_min_low_consistency[~quantile_min_low_consistency].index
# n2_data.loc[not_na_indexes, 'quantile_short_max_high'] = max_high_np_a[
#     np.arange(len(n2_data.loc[not_na_indexes])), n2_data.loc[not_na_indexes, 'min_low_quantile'].astype(int)]
# # n2_data.loc[consistent_quantile_min_low, 'pre_quantile_short_max_high'] = max_high_np_a[
# #     np.arange(len(consistent_quantile_min_low)), n2_data.loc[consistent_quantile_min_low, 'min_low_quantile'].astype(
# #         int) - 1]
# # n2_data.loc[consistent_quantile_min_low, 'short_drawdown_consistency'] = (
# #         n2_data.loc[consistent_quantile_min_low, 'pre_quantile_short_max_high']
# #         == n2_data.loc[consistent_quantile_min_low, 'quantile_short_max_high'])
# n2_data['short_drawdown'] = \
#     (n2_data['quantile_short_max_high'] - n2_data['worst_short_open']) / n2_data['worst_short_open']
# # n2_data[['High', 'Low', 'min_low', 'min_low_quantile', 'quantile_short_max_high', 'short_drawdown']]

In [ ]:
# risk_free_daily_rate = 0.1 / 365  # 10% yearly return
# order_fee = 0.005  # 0.1 / 365  # 10% yearly return
# n2_data['long_profit'] = n2_data['max_high'] - n2_data['worst_long_open']
# n2_data['short_profit'] = n2_data['worst_short_open'] - n2_data['min_low']
# n2_data['weighted_long_profit'] = \
#     n2_data['long_profit'] / n2_data['Close'] - n2_data['max_high_distance'] * risk_free_daily_rate - order_fee
# n2_data['weighted_short_profit'] = \
#     n2_data['short_profit'] / n2_data['Close'] - n2_data['min_low_distance'] * risk_free_daily_rate - order_fee
# n2_data['long_risk'] = (n2_data['long_drawdown'] / n2_data['weighted_long_profit'])
# n2_data['short_risk'] = (n2_data['short_drawdown'] / n2_data['weighted_short_profit'])
# max_risk = 1
# loser_shorts = n2_data[(n2_data['weighted_short_profit'] <= 0) | (n2_data['short_risk'] > max_risk)].index
# loser_longs = n2_data[(n2_data['weighted_long_profit'] <= 0) | (n2_data['long_risk'] > max_risk)].index
# n2_data.loc[loser_shorts, 'short_risk'] = max_risk
# n2_data.loc[loser_longs, 'long_risk'] = max_risk
# n2_data['long_signal'] = (1 - n2_data['long_risk'].fillna(1)) * n2_data['weighted_long_profit'].fillna(0)
# # n2_data['long_signal'] =  (n2_data['long_signal']) / ta.ema(n2_data['long_signal'], length=position_max_days)
# n2_data['short_signal'] = (1 - n2_data['short_risk'].fillna(1)) * n2_data['weighted_short_profit'].fillna(0)
# # n2_data['short_signal'] =  (n2_data['short_signal']) / ta.ema(n2_data['short_signal'], length=position_max_days)

In [ ]:
# from plotly.subplots import make_subplots
# 
# t = no_gap_data
# fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.01, row_heights=[0.5, 0.5, 0.1])
# fig.add_trace(go.Candlestick(x=t.index, open=t['Open'], close=t['Close'], high=t['High'], low=t['Low']), row=2, col=1)
# fig.add_scatter(x=t.index, y=t['max_high'], mode='lines', line=dict(color='red', width=1), name='max_high', row=2,
#                 col=1)
# fig.add_scatter(x=t.index, y=t['min_low'], mode='lines', line=dict(color='blue', width=1), name='min_low', row=2, col=1)
# fig.add_scatter(x=t.index, y=t['quantile_short_max_high'], mode='lines', line=dict(color='magenta', width=1),
#                 name='quantile_short_max_high', row=2, col=1)
# fig.add_scatter(x=t.index, y=t['short_risk'], mode='lines', line=dict(color='yellow', width=1),
#                 name='short_risk')
# fig.add_scatter(x=t.index, y=t['short_drawdown'], mode='lines', line=dict(color='pink', width=1), name='short_drawdown')
# fig.add_scatter(x=t.index, y=t['weighted_short_profit'], mode='lines', line=dict(color='blue', width=1),
#                 name='weighted_short_profit')
# fig.add_scatter(x=t.index, y=t['short_signal'], mode='lines', line=dict(color='green', width=1),
#                 name='short_signal')
# fig.update_layout(dragmode="zoom", yaxis2=dict(fixedrange=False), margin=dict(l=0, r=0, t=0, b=0), ).show()

In [ ]:
# from plotly.subplots import make_subplots
# 
# t = no_gap_data
# fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.01, row_heights=[0.6, 0.6])
# fig.add_trace(go.Candlestick(x=t.index, open=t['Open'], close=t['Close'], high=t['High'], low=t['Low']), row=2, col=1)
# fig.add_scatter(x=t.index, y=t['max_high'], mode='lines', line=dict(color='blue', width=1), name='max_high', row=2,
#                 col=1)
# fig.add_scatter(x=t.index, y=t['min_low'], mode='lines', line=dict(color='red', width=1), name='min_low', row=2, col=1)
# fig.add_scatter(x=t.index, y=t['quantile_long_min_low'], mode='lines', line=dict(color='magenta', width=1),
#                 name='quantile_long_min_low', row=2, col=1)
# fig.add_scatter(x=t.index, y=np.minimum(t['long_risk'], max_risk), mode='lines', line=dict(color='yellow', width=1),
#                 name='long_risk')
# fig.add_scatter(x=t.index, y=t['long_drawdown'], mode='lines', line=dict(color='pink', width=1), name='long_drawdown')
# fig.add_scatter(x=t.index, y=t['weighted_long_profit'], mode='lines', line=dict(color='blue', width=1),
#                 name='weighted_long_profit')
# fig.add_scatter(x=t.index, y=t['long_signal'], mode='lines', line=dict(color='green', width=1),
#                 name='long_signal')
# fig.update_layout(dragmode="zoom", yaxis2=dict(fixedrange=False), margin=dict(l=0, r=0, t=0, b=0), ).show()

In [ ]:
# number_of_samples = 10
# start = n2_data.index.min()
# end = n2_data.index.max()
# targets = pd.date_range(start, end, freq=(end - start) / number_of_samples)
# t_df = n2_data.dropna()
# samples = t_df.reset_index().sort_values(by='Date')['Date'].iloc[
#     [i for i in range(0, len(t_df), int(len(t_df) / number_of_samples))]].to_list()
# samples_df = n2_data.loc[samples]
# samples_df

In [ ]:
# from datetime import timedelta
# from plotly.subplots import make_subplots
# 
# t = no_gap_data
# fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.01, row_heights=[0.3, 0.7, 0.3], )
# fig.add_scatter(x=t.index, y=t['long_profit'], mode='lines', line=dict(color='blue', width=1), name='long_profit')
# fig.add_scatter(x=t.index, y=t['short_profit'], mode='lines', line=dict(color='red', width=1), name='short_profit')
# fig.add_scatter(x=t.index, y=1000*t['long_risk'], mode='lines', line=dict(color='LightSkyBlue', width=1),
#                 name='long_risk')
# fig.add_scatter(x=t.index, y=1000*t['short_risk'], mode='lines', line=dict(color='yellow', width=1),
#                 name='short_risk')
# fig.add_scatter(x=t.index, y=1000*t['long_signal'], mode='lines', line=dict(color='green', width=1),
#                 name='long_signal')
# fig.add_scatter(x=t.index, y=1000*t['short_signal'], mode='lines', line=dict(color='orange', width=1),
#                 name='short_signal')
# fig.add_scatter(x=t.index, y=t['min_low'], mode='lines', line=dict(color='red', width=1), name='min_low', row=2, col=1)
# fig.add_trace(
#     go.Candlestick(x=t.index, open=t['Open'], close=t['Close'], high=t['High'], low=t['Low']), row=2, col=1)
# fig.add_scatter(x=t.index, y=t['max_high'], mode='lines', line=dict(color='blue', width=1), name='max_high', row=2,
#                 col=1)
# 
# showlegend = True
# # for sample in samples:
# #     # x = [sample.replace(tzinfo=None), sample.replace(tzinfo=None) + position_max_days * timedelta(days=1)]
# #     # y = [float(t.loc[sample, 'Close']), float(t.loc[sample, 'Close'])]
# #     # fig.add_scatter(x=x, y=y, mode='lines', line=dict(color='gray', width=1), name='position_max_days', row=2, col=1,
# #     #                 legendgroup='position_max_days', showlegend=showlegend)
# #     x = [sample.replace(tzinfo=None),
# #          sample.replace(tzinfo=None) + t.loc[sample, 'max_high_distance'] * timedelta(days=1)]
# #     y = [float(t.loc[sample, 'High']), float(t.loc[sample, 'max_high'])]
# #     fig.add_scatter(x=x, y=y, mode='lines', line=dict(color='cyan', width=1), name='max_high', row=2, col=1,
# #                     legendgroup='max_high', showlegend=showlegend)
# #     x = [sample.replace(tzinfo=None),
# #          sample.replace(tzinfo=None) + t.loc[sample, 'min_low_distance'] * timedelta(days=1)]
# #     y = [float(t.loc[sample, 'Low']), float(t.loc[sample, 'min_low'])]
# #     fig.add_scatter(x=x, y=y, mode='lines', line=dict(color='pink', width=1), name='min_low', row=2, col=1,
# #                     legendgroup='min_low', showlegend=showlegend)
# #     # x = [sample.replace(tzinfo=None),
# #     #      sample.replace(tzinfo=None) + t.loc[sample, 'min_low_distance'] * timedelta(days=1)]
# #     # y = [float(t.loc[sample, 'worst_short_open']),
# #     #      float(t.loc[sample, 'worst_short_open'] + t.loc[sample, 'short_drawdown'])]
# #     # fig.add_scatter(x=x, y=y, mode='lines', line=dict(color='orange', width=1), name='short_drawdown', row=2, col=1,
# #     #                 legendgroup='short_drawdown', showlegend=showlegend)
# fig.update_layout(dragmode="zoom", yaxis2=dict(title="Price", fixedrange=False),
#                   margin=dict(l=0, r=0, t=10, b=10), ).show()

In [ ]:
# with_min_low_distance = no_gap_data[no_gap_data['min_low_distance'].notna()].index
# no_gap_data.loc[with_min_low_distance, ['High']]

In [ ]:
# with_min_low_distance = no_gap_data[no_gap_data['min_low_distance'].notna()].index
# no_gap_data.loc[with_min_low_distance, 'High']
# no_gap_data.loc[with_min_low_distance, 'min_low_distance'].astype(int)

In [ ]:
# with_min_low_distance = no_gap_data[no_gap_data['min_low_distance'].notna()].index
# t = no_gap_data.loc[with_min_low_distance, ['min_low_distance']].astype(int)
# t[t['min_low_distance'].isna()]
# no_gap_data.loc[with_min_low_distance, 'min_low_distance'].astype(int)